# Practical prompt development

Copyright (c) Patrick Hall, 2026--27. Released under the MIT License.

This notebook compares zero-shot and few-shot prompts for a small support-routing task, renders a reusable template, and records a controlled prompt revision. 

## 1. Configure the OpenAI client

The client searches upward for the repository `.env`. If no key is present, `getpass` requests it without displaying the value.

In [8]:
# 1. Configure the OpenAI client
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This O(depth) search lets the notebook run from its folder or the repository root.
project_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env').exists()), Path.cwd())
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
client = OpenAI(api_key=api_key)
TEXT_MODEL = os.getenv('OPENAI_TEXT_MODEL', 'gpt-4o-mini')
MAX_OUTPUT_TOKENS = int(os.getenv('OPENAI_MAX_OUTPUT_TOKENS', '128'))
print(f'Configured text model: {TEXT_MODEL}; output budget: {MAX_OUTPUT_TOKENS}')

Configured text model: gpt-4o-mini; output budget: 128


## 2. Define a small, transparent routing task

Each request has a human-authored expected label. Keep a held-out subset separate from the few-shot examples so a prompt is not evaluated only on demonstrations it has already seen.

In [9]:
# 2. Define a small, transparent routing task
ROUTES = ('billing', 'access', 'technical', 'other')
few_shot_examples = [
    ('I was billed twice, but I cannot sign in to review it.', 'access'),
    ('The password-reset email is not arriving, so I cannot view an unexpected charge.', 'access'),
    ('The billing page fails to open, so I cannot review a charge.', 'technical'),
    ('A charge is incorrect, but my account and billing page work.', 'billing'),
]
held_out_cases = [
    {'text': 'I need to review an extra payment, but I cannot reset my password.', 'expected': 'access'},
    {'text': 'My billing page crashes before I can inspect a duplicate charge.', 'expected': 'technical'},
    {'text': 'My account works, but my invoice has the wrong total.', 'expected': 'billing'},
    {'text': 'Where can I find the office mailing address?', 'expected': 'other'},
]
print(f'{len(few_shot_examples)} demonstrations; {len(held_out_cases)} held-out cases; routes: {ROUTES}')

4 demonstrations; 4 held-out cases; routes: ('billing', 'access', 'technical', 'other')


## 3. Render versioned zero-shot and few-shot templates

The template makes changeable components explicit. A version name, model name, and fixed label set are recorded with results so later comparisons are interpretable.

In [10]:
# 3. Render versioned zero-shot and few-shot templates
ZERO_SHOT_VERSION = 'routing-zero-shot-v2'
FEW_SHOT_VERSION = 'routing-few-shot-v2'

def render_prompt(text, examples=()):
    """Render a prompt for k few-shot demonstrations."""
    demonstrations = '\n'.join(f'Input: {example_text}\nRoute: {label}' for example_text, label in examples)
    example_block = f'\nExamples:\n{demonstrations}\n' if demonstrations else ''
    return f'''Classify the support request into exactly one route: {', '.join(ROUTES)}.
Return only the route label. If the request does not fit, return other.
{example_block}
Request: {text}
Route:'''.strip()

print(render_prompt(held_out_cases[0]['text'], few_shot_examples))

Classify the support request into exactly one route: billing, access, technical, other.
Return only the route label. If the request does not fit, return other.

Examples:
Input: I was billed twice, but I cannot sign in to review it.
Route: access
Input: The password-reset email is not arriving, so I cannot view an unexpected charge.
Route: access
Input: The billing page fails to open, so I cannot review a charge.
Route: technical
Input: A charge is incorrect, but my account and billing page work.
Route: billing

Request: I need to review an extra payment, but I cannot reset my password.
Route:


## 4. Call the model and validate its narrow output contract

The parser accepts only known labels. Retaining raw output makes a malformed response visible instead of silently converting it into a plausible result.

In [11]:
# 4. Call the model and validate its narrow output contract
def classify_request(prompt):
    """Make one API call."""
    response = client.responses.create(
        model=TEXT_MODEL,
        input=prompt,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    if response.status != 'completed':
        reason = getattr(response.incomplete_details, 'reason', response.status)
        raise RuntimeError(f'Response did not complete: {reason}')
    raw_text = (response.output_text or '').strip()
    normalized = raw_text.lower().strip(' .`')
    label = normalized if normalized in ROUTES else None
    return {'raw_output': raw_text, 'label': label}

example_result = classify_request(render_prompt(held_out_cases[0]['text']))
example_result

{'raw_output': 'billing', 'label': 'billing'}

## 5. Compare prompt versions on the same held-out cases

This is a compact controlled comparison: only the demonstration block changes. The sample is too small for a broad performance claim, but it can expose format failures and concrete disagreements worth reviewing.

In [12]:
# 5. Compare prompt versions on the same held-out cases
def evaluate_version(version, examples):
    """Run API calls and local scoring for n held-out cases."""
    rows = []
    for case in held_out_cases:
        result = classify_request(render_prompt(case['text'], examples))
        rows.append({
            'version': version,
            'text': case['text'],
            'expected': case['expected'],
            'predicted': result['label'],
            'raw_output': result['raw_output'],
            'correct': result['label'] == case['expected'],
        })
    return rows

zero_shot_rows = evaluate_version(ZERO_SHOT_VERSION, ())
few_shot_rows = evaluate_version(FEW_SHOT_VERSION, few_shot_examples)
results = zero_shot_rows + few_shot_rows
for version in (ZERO_SHOT_VERSION, FEW_SHOT_VERSION):
    version_rows = [row for row in results if row['version'] == version]
    accuracy = sum(row['correct'] for row in version_rows) / len(version_rows)
    valid_rate = sum(row['predicted'] is not None for row in version_rows) / len(version_rows)
    print(f'{version}: accuracy={accuracy:.0%}; valid-label rate={valid_rate:.0%}')
results

routing-zero-shot-v2: accuracy=50%; valid-label rate=100%
routing-few-shot-v2: accuracy=100%; valid-label rate=100%


[{'version': 'routing-zero-shot-v2',
  'text': 'I need to review an extra payment, but I cannot reset my password.',
  'expected': 'access',
  'predicted': 'billing',
  'raw_output': 'billing',
  'correct': False},
 {'version': 'routing-zero-shot-v2',
  'text': 'My billing page crashes before I can inspect a duplicate charge.',
  'expected': 'technical',
  'predicted': 'billing',
  'raw_output': 'billing',
  'correct': False},
 {'version': 'routing-zero-shot-v2',
  'text': 'My account works, but my invoice has the wrong total.',
  'expected': 'billing',
  'predicted': 'billing',
  'raw_output': 'billing',
  'correct': True},
 {'version': 'routing-zero-shot-v2',
  'text': 'Where can I find the office mailing address?',
  'expected': 'other',
  'predicted': 'other',
  'raw_output': 'other',
  'correct': True},
 {'version': 'routing-few-shot-v2',
  'text': 'I need to review an extra payment, but I cannot reset my password.',
  'expected': 'access',
  'predicted': 'access',
  'raw_output':

## 6. Record a refinement decision

A prompt change is not automatically an improvement. Record the hypothesis, exact change, test cases, model, result, and a reviewer decision. This creates a small prompt-library entry that can later be reproduced or rolled back.

In [13]:
# 6. Record a refinement decision
experiment_record = {
    'purpose': 'Route short support requests into one of four queues.',
    'baseline_version': ZERO_SHOT_VERSION,
    'candidate_version': FEW_SHOT_VERSION,
    'changed_component': 'Four input/output demonstrations were added.',
    'model': TEXT_MODEL,
    'test_set': 'Four held-out support requests defined in cell 2.',
    'success_criteria': 'Valid-label rate and human-reviewed routing accuracy.',
    'decision': 'Review rows, then promote, revise, or retain the baseline.',
}
experiment_record

{'purpose': 'Route short support requests into one of four queues.',
 'baseline_version': 'routing-zero-shot-v2',
 'candidate_version': 'routing-few-shot-v2',
 'changed_component': 'Four input/output demonstrations were added.',
 'model': 'gpt-4o-mini',
 'test_set': 'Four held-out support requests defined in cell 2.',
 'success_criteria': 'Valid-label rate and human-reviewed routing accuracy.',
 'decision': 'Review rows, then promote, revise, or retain the baseline.'}

## 7. Interpret results and limits

Few-shot examples can clarify labels and output format, but they do not prove reliability. Expand the test set before deployment, inspect errors, preserve prompt versions, and monitor outcomes after release. 

In [14]:
# 7. Summarize the comparison for review
for row in results:
    status = 'correct' if row['correct'] else 'review'
    print(f"{row['version']}: {status} | expected={row['expected']} | predicted={row['predicted']} | {row['text']}")

print('Next step: add representative cases and review failures before using either version operationally.')

routing-zero-shot-v2: review | expected=access | predicted=billing | I need to review an extra payment, but I cannot reset my password.
routing-zero-shot-v2: review | expected=technical | predicted=billing | My billing page crashes before I can inspect a duplicate charge.
routing-zero-shot-v2: correct | expected=billing | predicted=billing | My account works, but my invoice has the wrong total.
routing-zero-shot-v2: correct | expected=other | predicted=other | Where can I find the office mailing address?
routing-few-shot-v2: correct | expected=access | predicted=access | I need to review an extra payment, but I cannot reset my password.
routing-few-shot-v2: correct | expected=technical | predicted=technical | My billing page crashes before I can inspect a duplicate charge.
routing-few-shot-v2: correct | expected=billing | predicted=billing | My account works, but my invoice has the wrong total.
routing-few-shot-v2: correct | expected=other | predicted=other | Where can I find the offic